In [ ]:
# --- bootstrap: make src/ importable and run from the repository root ---
import os, sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "src" / "path_info.py").exists())
sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)

In [ ]:
# !pip install adjustText

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, Subset
import torch.nn.functional as F
from torchvision import transforms, datasets
from transformers import BlipModel, BlipProcessor
from tqdm import tqdm
import pandas as pd
import os
from sklearn.model_selection import StratifiedKFold
import numpy as np
import glob
from PIL import Image
from sklearn.metrics import f1_score, accuracy_score, classification_report
import re
from functools import lru_cache
import multiprocessing
import mmap
import json

from transformers import BlipModel, BlipProcessor, CLIPModel, CLIPProcessor

from baseline_model import CustomClassifier, plot_history
from CBM_model import CBM_model, plot_concept_to_class_weights

In [ ]:
import warnings

warnings.filterwarnings("ignore", category=FutureWarning, module="huggingface_hub.file_download")

os.environ["TOKENIZERS_PARALLELISM"] = "false" # to remove warnings about parallelism :
# huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
# To disable this warning, you can either:
# - Avoid using `tokenizers` before the fork if possible
# - Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)

In [ ]:
from CBM_pipeline import batch_size, device, num_workers, max_len

# N24

In [ ]:
from CBM_pipeline import run_CBM
from path_info import PATH

backbone = 'clip'
concept_scores = f'{PATH}/datasets/N24News/combine/cb_llm_annotation/{backbone}_multimodal/outputs_concept_scoring/blue_checkpoints/{backbone}/cavs/regression/sorted_macro_concepts_coverage_MJ_cb_llm_abs_all_LIG.pkl'

In [ ]:
# no heuristique, no KAN, no leakage loss

model = run_CBM(dataset='N24', dataset_type='CBLLM', combine_type='combine', backbone='clip', concept_representation='importance', num_epochs=10, load=False, plot=True, leakage_loss=False, kan_layer=False)

In [ ]:
# heuristique, no KAN, no leakage loss

# choose minimum concept level so that coverage is at least 99%

model = run_CBM(dataset='N24', dataset_type='CBLLM', combine_type='combine', backbone='clip', concept_representation='importance', num_epochs=10, import_concept_list=concept_scores, concept_level=12, load=False, plot=True, leakage_loss=False, kan_layer=False)

In [ ]:
# no heuristique, KAN, no leakage loss

model = run_CBM(dataset='N24', dataset_type='CBLLM', combine_type='combine', backbone='clip', concept_representation='importance', num_epochs=10, load=False, plot=True, leakage_loss=False, kan_layer=True)

In [ ]:
# heuristique, KAN, no leakage loss

# choose minimum concept level so that coverage is at least 99%

model = run_CBM(dataset='N24', dataset_type='CBLLM', combine_type='combine', backbone='clip', concept_representation='importance', num_epochs=10, import_concept_list=concept_scores, concept_level=12, load=False, plot=True, leakage_loss=False, kan_layer=True)

In [ ]:
# no heuristique, no KAN, leakage loss

model = run_CBM(dataset='N24', dataset_type='CBLLM', combine_type='combine', backbone='clip', concept_representation='importance', num_epochs=10, load=False, plot=True, leakage_loss=True, leakage_loss_activation='up', kan_layer=False)

In [ ]:
# heuristique, no KAN, leakage loss

# choose minimum concept level so that coverage is at least 99%

model = run_CBM(dataset='N24', dataset_type='CBLLM', combine_type='combine', backbone='clip', concept_representation='importance', num_epochs=10, import_concept_list=concept_scores, concept_level=12, load=False, plot=True, leakage_loss=True, leakage_loss_activation='up', kan_layer=False)

In [ ]:
# no heuristique, KAN, leakage loss

model = run_CBM(dataset='N24', dataset_type='CBLLM', combine_type='combine', backbone='clip', concept_representation='importance', num_epochs=10, load=False, plot=True, leakage_loss=True, leakage_loss_activation='up', kan_layer=True)

In [ ]:
# heuristique, KAN, leakage loss

# choose minimum concept level so that coverage is at least 99%

model = run_CBM(dataset='N24', dataset_type='CBLLM', combine_type='combine', backbone='clip', concept_representation='importance', num_epochs=10, import_concept_list=concept_scores, concept_level=12, load=False, plot=True, leakage_loss=True, leakage_loss_activation='up', kan_layer=True)

# Tests varying the number of concepts

In [ ]:
# heuristique, no KAN, no leakage loss

# tests with specific list of concepts

# first list of concepts, taking into account coverage

heur_concept_list = [
"food origins",
"recipes",
"ethical sourcing",
"artist interviews",
"lyrical analysis",
"cultural traditions",
"food science",
"nutritional composition",
"heritage perspectives",
"ingredients",
"championships",
"artist management",
"artist collaborations",
"athlete profiles",
"composition credits",
"historical context",
"commentary",
"cryptocurrency",
"venture capital",
"taste experiences",
"fan culture",
"music education",
"innovation narratives",
"album reviews",
"innovation stories",
"recipe guides",
"career milestones",
"dietary patterns",
"fermentation",
"industry trends",
"cultural significance",
"controversies",
"catalog retrospectives",
"sound design",
"sports medicine",
"team sports",
"tournaments"
]

print(len(heur_concept_list))

history, model = run_CBM(dataset='N24', dataset_type='CBLLM', combine_type='combine', backbone='clip', concept_representation='importance', num_epochs=10, load=False, plot=True, leakage_loss=False, kan_layer=False, select_concepts=heur_concept_list)

print(history)

In [ ]:
# heuristique, no KAN, no leakage loss

# tests with specific list of concepts

# second with best R² only

heur_concept_list = [
    "recipes",
    "recipe guides",
    "cuisine trends",
    "gastronomic techniques",
    "food science",
    "food origins",
    "music education",
    "specialty diets",
    "food technology",
    "dietary patterns",
    "cooking techniques",
    "music theory",
    "musical composition",
    "album releases",
    "restaurant developments",
    "food festivals",
    "music publishers",
    "restaurant reviews",
    "restaurants",
    "culinary awards",
    "artist interviews",
    "soundtrack releases",
    "tech corporations",
    "culinary institutes",
    "ingredients",
    "food safety",
    "album reviews",
    "music festivals",
    "nutritional composition",
    "performance venues",
    "artist collaborations",
    "record labels",
    "concert tours",
    "tasting evaluations",
    "flavor profiles",
    "ingredient features",
    "genre evolution",
    "food manufacturers",
]


print(len(heur_concept_list))

history, model = run_CBM(dataset='N24', dataset_type='CBLLM', combine_type='combine', backbone='clip', concept_representation='importance', num_epochs=10, load=False, plot=True, leakage_loss=False, kan_layer=False, select_concepts=heur_concept_list)

print(history)

In [ ]:
# heuristique, no KAN, no leakage loss

# tests with random concepts

history, model = run_CBM(dataset='N24', dataset_type='CBLLM', combine_type='combine', backbone='clip', concept_representation='importance', num_epochs=10, load=False, plot=True, leakage_loss=False, kan_layer=False, random_concepts=38)

print(history)

In [ ]:
# heuristique, no KAN, no leakage loss

# tests with specific list of concepts

history, model = run_CBM(dataset='N24', dataset_type='CBLLM', combine_type='combine', backbone='clip', concept_representation='importance', num_epochs=10, load=False, plot=True, leakage_loss=False, kan_layer=False, random_concepts=76)

print(history)

In [ ]:
# heuristique, no KAN, no leakage loss

# tests with specific list of concepts

history, model = run_CBM(dataset='N24', dataset_type='CBLLM', combine_type='combine', backbone='clip', concept_representation='importance', num_epochs=10, load=False, plot=True, leakage_loss=False, kan_layer=False, random_concepts=114)

print(history)

In [ ]:
# heuristique, no KAN, no leakage loss

# tests with specific list of concepts

history, model = run_CBM(dataset='N24', dataset_type='CBLLM', combine_type='combine', backbone='clip', concept_representation='importance', num_epochs=10, load=False, plot=True, leakage_loss=False, kan_layer=False, random_concepts=152)

print(history)

In [ ]:
# heuristique, no KAN, no leakage loss

# tests with specific list of concepts

history, model = run_CBM(dataset='N24', dataset_type='CBLLM', combine_type='combine', backbone='clip', concept_representation='importance', num_epochs=10, load=False, plot=True, leakage_loss=False, kan_layer=False)

print(history)

# Random concepts

In [ ]:
# no heuristique, no KAN, no leakage loss

# tests with random concepts (5 runs: each run draws a different random subset of 38 concepts)

for _ in range(5):
    history, model = run_CBM(dataset='N24', dataset_type='CBLLM', combine_type='combine', backbone='clip', concept_representation='importance', num_epochs=10, load=False, plot=True, leakage_loss=False, kan_layer=False, random_concepts=38)

    print(history)

# Random concepts KAN + loss leakage

In [ ]:
# no heuristique, KAN, leakage loss

# tests with random concepts (8 runs: each run draws a different random subset of 38 concepts)

for _ in range(8):
    history, model = run_CBM(dataset='N24', dataset_type='CBLLM', combine_type='combine', backbone='clip', concept_representation='importance', num_epochs=10, load=False, plot=True, leakage_loss=True, leakage_loss_activation='up', kan_layer=True, random_concepts=38)

    print(history)